# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR^2 dataset package using the `mlcroissant` library. 

### Dataset Source
The dataset is described using a Croissant schema and is accessible via the following URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` and dependencies are installed
!pip install mlcroissant pandas

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant metadata schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Instantiate the dataset object
dataset = mlc.Dataset(croissant_url)
# Display general metadata about the dataset
print(f"Dataset Name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")

## 2. Data Overview
Let's review the available record sets and their fields using their `@id` values as required by the Croissant schema.

In [ ]:
# List all available RecordSet @ids in the dataset
print("Available Record Sets and their fields (by @id):")
record_set_infos = []
for record_set in dataset.record_sets:
    print(f"- Record Set: {record_set['@id']}")
    fields = record_set.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    for field in fields:
        if isinstance(field, dict):
            print(f"    - Field: {field.get('@id', field)}")
        else:
            # Sometimes the list is just a list of @ids
            print(f"    - Field: {field}")
    record_set_infos.append({
        'record_set_id': record_set['@id'],
        'fields': [field.get('@id', field) if isinstance(field, dict) else field for field in fields],
    })

if not record_set_infos:
    print('No record sets found in the dataset. Check Croissant schema or dataset availability.')

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Refer to record set and field `@id` values identified above.

**Note:** If your dataset has multiple record sets, this code will extract all available record sets to separate Pandas DataFrames, using the `@id` as the key.

In [ ]:
# Prepare to extract each RecordSet found
dataframes = {}

record_set_ids = [info['record_set_id'] for info in record_set_infos]
if not record_set_ids:
    print('No RecordSets present in Croissant schema. Data extraction cannot proceed.')
else:
    for record_set_id in record_set_ids:
        try:
            records = list(dataset.records(record_set=record_set_id))
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded RecordSet '{record_set_id}' with shape {df.shape}.")
        except Exception as e:
            print(f"Could not load records for RecordSet {record_set_id}: {e}")

    # Display first DataFrame info if data loaded
    if dataframes:
        first_rs_id = next(iter(dataframes.keys()))
        print(f"\nColumns in RecordSet '{first_rs_id}':")
        print(dataframes[first_rs_id].columns.tolist())
        display(dataframes[first_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Let's conduct some basic EDA: filter, normalize, and group on a numeric field. 

- **Note:** If your RecordSet or numeric fields were empty, use mock data or adjust the cell according to actual columns loaded above.
- *Always reference columns and fields by their `@id` where possible.*

In [ ]:
# For demonstration, select the first loaded RecordSet and a numeric field if present
if dataframes:
    record_set_id = next(iter(dataframes.keys()))
    df = dataframes[record_set_id].copy()
    print(f"Using RecordSet: {record_set_id}")
    
    # Attempt to automatically detect a numeric field (float or int columns)
    numeric_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if numeric_candidates:
        numeric_field_id = numeric_candidates[0]
        print(f"Selected numeric field for EDA: {numeric_field_id}")
        
        threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 0
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        )
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Attempt to group by a non-numeric field
        group_candidates = [col for col in df.columns if not pd.api.types.is_numeric_dtype(df[col])]
        if group_candidates:
            group_field_id = group_candidates[0]
            print(f"Grouping by field: {group_field_id}")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(f"Grouped means:")
            display(grouped_df.head())
        else:
            print("No non-numeric fields available for grouping in RecordSet.")
    else:
        print("No numeric fields detected in RecordSet for EDA.")
else:
    print("No DataFrames available to perform EDA.")

## 5. Visualization
Visualize a selected numeric field as a histogram if available.

- You can replace or enhance the plot below with more advanced plots as appropriate for your data.

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

if dataframes:
    first_rs_id = next(iter(dataframes.keys()))
    df = dataframes[first_rs_id]
    # Find a numeric column
    numeric_cols = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if numeric_cols:
        plt.figure(figsize=(8, 5))
        df[numeric_cols[0]].plot(kind='hist', bins=20, alpha=0.7)
        plt.title(f"Distribution of {numeric_cols[0]} in RecordSet {first_rs_id}")
        plt.xlabel(numeric_cols[0])
        plt.ylabel("Count")
        plt.grid(True)
        plt.show()
    else:
        print("No numeric field to visualize in this RecordSet.")
else:
    print("No DataFrames loaded for visualization.")

## 6. Conclusion
This notebook demonstrated how to load and explore a Croissant-structured dataset using the `mlcroissant` library:
- The dataset's metadata describes ordered logistic regression results for predictors of indigenous and modern knowledge adoption in rangeland management.
- Data was loaded, and available record sets and their fields were listed and accessed by `@id`, ensuring reproducible references.
- We performed simple exploratory analysis (EDA) and basic visualization using numeric fields (where available).

You can extend this notebook with further domain-specific analyses, advanced data cleaning, or model-building workflows as needed for your research or implementation.